# 🧠 State & Memory Management for Long-Running Investigations

**Goal:** Master conversation persistence and context management so agents maintain investigation context across sessions, shifts, and restarts.

```
Day 1 (Analyst A)           Day 2 (Analyst B)           Day 3 (Analyst A)
─────────────────           ─────────────────           ─────────────────
conversation_id=abc  ─────► Load conversation abc ─────► Load conversation abc
Initial triage       │      Continue analysis      │      Closure + report
Save conversation_id │      Full context           │      All history intact
                     └────────────────────────────┘
```

**Key insight:** Conversation IDs ARE your investigation state. Persist them — the Foundry service keeps the full message history server-side. Use `openai_client.conversations.create()` to start, `responses.create(conversation=conversation_id, ...)` to continue.

**SDK:** `azure-ai-projects>=2.0.0` — Responses API conversations (NOT Assistants API threads).


In [ ]:
!uv pip install "azure-ai-projects>=2.0.0" azure-identity python-dotenv pandas matplotlib seaborn -q


In [ ]:
import os, json
import datetime
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

load_dotenv(Path('..') / '.env', override=False)

AZURE_AI_PROJECT_ENDPOINT = os.getenv('AZURE_AI_PROJECT_ENDPOINT', '')
MODEL_DEPLOYMENT_NAME = os.getenv('MODEL_DEPLOYMENT_NAME', 'gpt-4.1')
# If you ran Notebook 01 and saved a conversation_id, paste it here:
EXISTING_CONVERSATION_ID = os.getenv('EXISTING_CONVERSATION_ID', '')

MOCK_MODE = not bool(AZURE_AI_PROJECT_ENDPOINT)
print('⚠️  MOCK MODE' if MOCK_MODE else f'✅ Connected: {AZURE_AI_PROJECT_ENDPOINT[:50]}...')


## 🔵 Concept: Conversation Lifecycle in a SOC Investigation

A **conversation** (`openai_client.conversations.create()`) is a persistent multi-turn session between an analyst and an agent. Every `responses.create(conversation=id, ...)` call adds to the history — all messages, tool calls, and responses are stored server-side. No context is lost between sessions or analyst shifts.

| Old SDK (1.x) | New SDK (2.x) |
|---|---|
| `agents_client.threads.create()` | `openai_client.conversations.create()` |
| `agents_client.messages.create(thread_id=...)` | Next `responses.create(conversation=id, input=...)` |
| `agents_client.runs.create_and_process(thread_id=..., agent_id=...)` | `openai_client.responses.create(conversation=id, extra_body={agent_reference:...})` |
| `msg.text_messages[-1].text.value` | `response.output_text` |
| `EXISTING_THREAD_ID` | `EXISTING_CONVERSATION_ID` |


In [ ]:
# Simulate a full investigation thread history
simulated_thread = [
    {'role': 'user',      'timestamp': '2025-03-07T02:20:00Z', 'content': 'Alert ALT-2025-002 received: Cobalt Strike beacon on 10.0.0.45. Investigate.', 'tokens': 45},
    {'role': 'assistant', 'timestamp': '2025-03-07T02:20:45Z', 'content': 'Delegated to Threat Intel Agent. IOC 194.165.16.101 confirmed APT29 C2. Delegated enrichment to MITRE mapping agent.', 'tokens': 128},
    {'role': 'user',      'timestamp': '2025-03-07T02:45:00Z', 'content': 'Host isolation command executed by IR team. What should we investigate next?', 'tokens': 22},
    {'role': 'assistant', 'timestamp': '2025-03-07T02:45:30Z', 'content': 'Post-isolation checklist: 1) Memory dump captured? 2) Review svc-backup account for lateral movement to domain controllers. 3) Check Azure AD for new OAuth grants from compromised host context.', 'tokens': 201},
    {'role': 'user',      'timestamp': '2025-03-07T09:15:00Z', 'content': 'Analyst B here (shift change). What is the current status of this investigation?', 'tokens': 18},
    {'role': 'assistant', 'timestamp': '2025-03-07T09:15:20Z', 'content': 'CASE SUMMARY [ALT-2025-002]: Host 10.0.0.45 isolated at 02:45Z. APT29 Cobalt Strike beacon confirmed. svc-backup account under investigation for lateral movement. Memory dump pending sandbox analysis. Priority: P1 — Active.', 'tokens': 245},
    {'role': 'user',      'timestamp': '2025-03-07T14:30:00Z', 'content': 'Sandbox flagged the memory dump as Cobalt Strike 4.9.1. IOC: beacon checksum SHA256=a3f8c7... Notify stakeholders and close containment phase.', 'tokens': 38},
    {'role': 'assistant', 'timestamp': '2025-03-07T14:31:00Z', 'content': 'Containment phase CLOSED. New IOC added: SHA256=a3f8c7... Escalating to stakeholder report. Transitioning to eradication phase.', 'tokens': 189},
]

df_thread = pd.DataFrame(simulated_thread)
df_thread['timestamp'] = pd.to_datetime(df_thread['timestamp'])
df_thread['time_str'] = df_thread['timestamp'].dt.strftime('%H:%M')
df_thread['role_icon'] = df_thread['role'].map({'user': '🧑 Analyst', 'assistant': '🤖 Agent'})

display(df_thread[['role_icon', 'time_str', 'tokens', 'content']]
        .rename(columns={'role_icon': 'Role', 'time_str': 'Time', 'tokens': 'Tokens', 'content': 'Message'})
        .style.map(lambda v: 'color: #4ecdc4' if '🤖' in str(v) else 'color: #ff9944' if '🧑' in str(v) else '',
                   subset=['Role'])
        .set_properties(**{'background-color': '#1e2127', 'color': '#e6e6e6'}))

In [ ]:
# Visualize investigation timeline with token usage
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), gridspec_kw={'height_ratios': [3, 1]})
fig.patch.set_facecolor('#0d1117')

# Timeline
ax1.set_facecolor('#0d1117')
for _, row in df_thread.iterrows():
    color = '#4ecdc4' if row['role'] == 'assistant' else '#ff9944'
    y_pos = 1 if row['role'] == 'assistant' else 2
    ax1.scatter(row['timestamp'], y_pos, color=color, s=150, zorder=5)
    ax1.text(row['timestamp'], y_pos + 0.15, row['time_str'], ha='center', fontsize=7, color=color)

ax1.axhline(y=1, color='#4ecdc433', linewidth=1, linestyle='--')
ax1.axhline(y=2, color='#ff994433', linewidth=1, linestyle='--')
ax1.set_yticks([1, 2])
ax1.set_yticklabels(['🤖 Agent', '🧑 Analyst'], color='white', fontsize=10)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax1.tick_params(colors='white')
ax1.set_title('Investigation Timeline — Thread Conversation Flow', color='white', pad=10)
ax1.set_ylim(0.5, 2.7)
for s in ax1.spines.values(): s.set_color('#333')

# Add shift change annotation
shift_time = pd.Timestamp('2025-03-07T09:15:00Z')
ax1.axvline(x=shift_time, color='#ffdd44', linewidth=1.5, linestyle=':')
ax1.text(shift_time, 2.5, '← Shift Change →', color='#ffdd44', ha='center', fontsize=8)

# Token usage bar
ax2.set_facecolor('#0d1117')
bar_colors = ['#4ecdc4' if r == 'assistant' else '#ff9944' for r in df_thread['role']]
ax2.bar(df_thread['timestamp'], df_thread['tokens'], color=bar_colors, width=pd.Timedelta('30min'), edgecolor='#333')
ax2.set_ylabel('Tokens', color='white', fontsize=8)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax2.tick_params(colors='white')
ax2.set_title('Token Usage per Message', color='white')
for s in ax2.spines.values(): s.set_color('#333')

total_tokens = df_thread['tokens'].sum()
ax2.text(0.98, 0.85, f'Total: {total_tokens} tokens', transform=ax2.transAxes,
         ha='right', color='white', fontsize=9, bbox=dict(facecolor='#333', edgecolor='#555', pad=4))

plt.tight_layout()
plt.show()

## 🔴 Create a Persistent Investigation Conversation

Each `openai_client.conversations.create()` gives you a persistent ID. Every subsequent `responses.create(conversation=id, ...)` appends to the same conversation history — the agent has full context automatically.


In [ ]:
if not MOCK_MODE:
    from azure.ai.projects import AIProjectClient
    from azure.ai.projects.models import PromptAgentDefinition
    from azure.identity import DefaultAzureCredential

    credential = DefaultAzureCredential()
    project_client = AIProjectClient(endpoint=AZURE_AI_PROJECT_ENDPOINT, credential=credential)
    openai_client = project_client.get_openai_client()

    soc_lead = project_client.agents.create_version(
        agent_name='soc-lead-stateful',
        definition=PromptAgentDefinition(
            model=MODEL_DEPLOYMENT_NAME,
            instructions=(
                'You are the SOC Lead for Case Management. You maintain investigation state '
                'across analyst shifts. When asked for a case summary, synthesize all prior '
                'conversation context into a concise SITREP (Situation Report) covering: '
                'current status, key findings, actions taken, and next steps.'
            ),
        ),
    )
    print(f'✅ Stateful SOC Lead agent: {soc_lead.name} v{soc_lead.version}')
else:
    print('⚠️  Skipping — set AZURE_AI_PROJECT_ENDPOINT')


In [ ]:
if not MOCK_MODE:
    AGENT_REF = {'agent_reference': {'name': soc_lead.name, 'type': 'agent_reference'}}

    # ── Session 1: Initial triage (Analyst A) ─────────────────────────────────
    if EXISTING_CONVERSATION_ID:
        # RESUME existing investigation — all conversation history persists server-side
        conversation_id = EXISTING_CONVERSATION_ID
        print(f'♻️  Resumed existing conversation: {conversation_id}')
    else:
        # Start fresh investigation
        conversation = openai_client.conversations.create()
        conversation_id = conversation.id
        print(f'🆕 New conversation: {conversation_id}')
        print(f'\n📌 SAVE THIS ID — paste into EXISTING_CONVERSATION_ID in .env to resume:')
        print(f'   EXISTING_CONVERSATION_ID={conversation_id}')

    # ── First analyst turn ─────────────────────────────────────────────────────
    response1 = openai_client.responses.create(
        conversation=conversation_id,
        input=(
            'Starting investigation for Incident INC-2025-042. Cobalt Strike beacon detected '
            'on host 10.0.0.45. Source IP 194.165.16.101 confirmed APT29 C2. '
            'Host isolated at 02:45Z. Memory dump initiated.'
        ),
        extra_body=AGENT_REF,
    )
    print(f'\n🤖 [Session 1 Response]:\n{response1.output_text}')
else:
    print('⚠️  Skipping — set AZURE_AI_PROJECT_ENDPOINT')


In [ ]:
if not MOCK_MODE:
    # ── Session 2: Shift change — Analyst B picks up the case ─────────────────
    # Analyst B uses the SAME conversation_id — full context is preserved server-side
    print(f'🔄 Resuming conversation: {conversation_id} (simulating shift change)\n')

    response2 = openai_client.responses.create(
        conversation=conversation_id,
        input='Analyst B here — taking over. What is the current SITREP for this case? What was done and what do I need to do next?',
        extra_body=AGENT_REF,
    )
    print(f'🤖 [SITREP for Analyst B]:\n{response2.output_text}')
else:
    print('⚠️  Skipping — set AZURE_AI_PROJECT_ENDPOINT')


## 🔴 List All Active Conversations

In production, store conversation IDs in a case management database (e.g., Cosmos DB, Azure SQL). Here we list active conversations using `openai_client.conversations.list()`.


In [ ]:
if not MOCK_MODE:
    conversations = list(openai_client.conversations.list())
    conv_data = []
    for c in conversations[:10]:  # show first 10
        conv_data.append({
            'conversation_id': c.id,
            'created_at': str(getattr(c, 'created_at', 'N/A')),
        })

    df_convs = pd.DataFrame(conv_data)
    display(df_convs.style.set_properties(**{'background-color': '#1e2127', 'color': '#e6e6e6'}))
    print(f'\nTotal conversations shown: {len(conversations[:10])}')
else:
    print('⚠️  Skipping conversation listing — connect to Azure')


## 🔵 Long-Term Memory Pattern

For investigations spanning weeks (ransomware, APT), thread context alone may not be enough. The **External Memory** pattern stores structured findings externally and injects relevant history as system context.

```
                     ┌──────────────────────────┐
                     │   External Memory Store   │
                     │  (Azure Cosmos DB / SQL)   │
                     │                           │
                     │  case_id → {              │
                     │    timeline: [...],       │
                     │    iocs: [...],           │
                     │    confirmed_ttp: [...],  │
                     │    thread_id: 'abc123'    │
                     │  }                        │
                     └───────────┬───────────────┘
                                 │  inject at session start
                                 ▼
                    Agent instructions + thread messages
```

In [ ]:
# 🔵 Long-Term Memory Store (works without credentials)

class SOCMemoryStore:
    """Lightweight in-memory store simulating Cosmos DB / Azure SQL for investigation state."""

    def __init__(self):
        self._store: dict = {}

    def open_case(self, case_id: str, alert_id: str, thread_id: str) -> dict:
        case = {
            'case_id': case_id,
            'alert_id': alert_id,
            'thread_id': thread_id,
            'opened_at': datetime.datetime.utcnow().isoformat(),
            'status': 'Active',
            'iocs': [],
            'confirmed_ttps': [],
            'actions_taken': [],
            'notes': []
        }
        self._store[case_id] = case
        return case

    def add_ioc(self, case_id: str, ioc: str, ioc_type: str, confidence: float):
        self._store[case_id]['iocs'].append({'value': ioc, 'type': ioc_type, 'confidence': confidence})

    def add_ttp(self, case_id: str, technique_id: str, technique_name: str):
        self._store[case_id]['confirmed_ttps'].append({'id': technique_id, 'name': technique_name})

    def log_action(self, case_id: str, action: str, analyst: str):
        self._store[case_id]['actions_taken'].append({
            'action': action, 'analyst': analyst,
            'at': datetime.datetime.utcnow().strftime('%H:%M:%S')
        })

    def get_context_injection(self, case_id: str) -> str:
        """Generate a context string to inject into agent instructions for continuity."""
        c = self._store.get(case_id, {})
        if not c:
            return 'No prior case context found.'
        ioc_str = ', '.join(f"{i['value']} ({i['type']}, {i['confidence']:.0%})" for i in c['iocs'])
        ttp_str = ', '.join(f"{t['id']}-{t['name']}" for t in c['confirmed_ttps'])
        action_str = '; '.join(f"[{a['at']}] {a['analyst']}: {a['action']}" for a in c['actions_taken'])
        return (
            f'CASE {case_id} CONTEXT:\n'
            f'  Status: {c["status"]} | Thread: {c["thread_id"]}\n'
            f'  Confirmed IOCs: {ioc_str or "none"}\n'
            f'  Confirmed TTPs: {ttp_str or "none"}\n'
            f'  Actions Taken: {action_str or "none"}'
        )

    def close_case(self, case_id: str, resolution: str):
        self._store[case_id]['status'] = 'Closed'
        self._store[case_id]['resolved_at'] = datetime.datetime.utcnow().isoformat()
        self._store[case_id]['resolution'] = resolution

    def get_case(self, case_id: str) -> dict:
        return self._store.get(case_id, {})


# Demo the memory store
memory = SOCMemoryStore()
memory.open_case('INC-2025-042', 'ALT-2025-002', 'thread_abc123')
memory.add_ioc('INC-2025-042', '194.165.16.101', 'IPv4/C2', 0.95)
memory.add_ioc('INC-2025-042', 'a3f8c7d2...', 'SHA256', 0.99)
memory.add_ttp('INC-2025-042', 'T1055.012', 'Process Hollowing')
memory.add_ttp('INC-2025-042', 'T1071.001', 'Web Protocols C2')
memory.log_action('INC-2025-042', 'Host 10.0.0.45 isolated', 'Analyst-A')
memory.log_action('INC-2025-042', 'Memory dump submitted to sandbox', 'Analyst-A')

print('📋 Context Injection for Analyst B:')
print('=' * 60)
print(memory.get_context_injection('INC-2025-042'))

In [ ]:
# Visualize case memory state
case = memory.get_case('INC-2025-042')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.patch.set_facecolor('#0d1117')
fig.suptitle(f'Case {case["case_id"]} — Memory State', color='white', fontsize=12)

categories = ['IOCs', 'TTPs', 'Actions']
counts = [len(case['iocs']), len(case['confirmed_ttps']), len(case['actions_taken'])]
max_capacity = [10, 8, 20]  # typical SOC case limits

colors = ['#ff6b6b', '#4ecdc4', '#45b7d1']
for ax, cat, count, cap, color in zip(axes, categories, counts, max_capacity, colors):
    ax.set_facecolor('#0d1117')
    ax.barh([cat], [count], color=color, height=0.5, edgecolor='#333')
    ax.barh([cat], [cap - count], left=[count], color=color + '22', height=0.5, edgecolor='#333')
    ax.set_xlim(0, cap)
    ax.text(count + 0.2, 0, f'{count}/{cap}', va='center', color='white', fontsize=11, fontweight='bold')
    ax.set_title(cat, color=color, fontsize=11)
    ax.tick_params(colors='white')
    for s in ax.spines.values(): s.set_color('#333')

plt.tight_layout()
plt.show()

# IOC confidence chart
if case['iocs']:
    fig2, ax = plt.subplots(figsize=(8, 2.5))
    fig2.patch.set_facecolor('#0d1117')
    ax.set_facecolor('#0d1117')
    ioc_labels = [f"{i['value'][:20]}\n({i['type']})" for i in case['iocs']]
    ioc_confs = [i['confidence'] for i in case['iocs']]
    ioc_colors = ['#ff4444' if c > 0.9 else '#ff9944' if c > 0.7 else '#44bbff' for c in ioc_confs]
    ax.barh(ioc_labels, ioc_confs, color=ioc_colors, height=0.4, edgecolor='#333')
    ax.set_xlim(0, 1)
    ax.set_xlabel('Confidence Score', color='white')
    ax.set_title('IOC Confidence Levels', color='white')
    ax.tick_params(colors='white')
    ax.axvline(x=0.7, color='#ffdd44', linewidth=1, linestyle='--')
    ax.text(0.71, -0.5, 'Threshold', color='#ffdd44', fontsize=8)
    for s in ax.spines.values(): s.set_color('#333')
    plt.tight_layout()
    plt.show()

## 🔴 Context Injection: Resume with Full Memory

In [ ]:
if not MOCK_MODE:
    # Build dynamic instructions with injected case context
    context_injection = memory.get_context_injection('INC-2025-042')

    # Create a session-specific agent with context pre-loaded into instructions
    case_aware_agent = project_client.agents.create_version(
        agent_name='soc-lead-context-aware',
        definition=PromptAgentDefinition(
            model=MODEL_DEPLOYMENT_NAME,
            instructions=(
                'You are the SOC Lead. You have deep institutional knowledge of this case.\n\n'
                f'{context_injection}\n\n'
                'Use this context to provide continuity for the incoming analyst. '
                'Never repeat information already known — only surface new insights.'
            ),
        ),
    )

    # Single-turn: Analyst B immediately has full case context
    response = openai_client.responses.create(
        input='Hi, Analyst B here. Just starting my shift. What should I focus on for INC-2025-042?',
        extra_body={'agent_reference': {'name': case_aware_agent.name, 'type': 'agent_reference'}},
    )
    print('🤖 Case-Aware Agent Response:')
    print('─' * 60)
    print(response.output_text)

    # Cleanup
    project_client.agents.delete_version(agent_name=case_aware_agent.name, agent_version=case_aware_agent.version)
    project_client.agents.delete_version(agent_name=soc_lead.name, agent_version=soc_lead.version)
    openai_client.close()
    project_client.close()
    print('\n✅ Agents deleted')
else:
    print('⚠️  Skipping — set AZURE_AI_PROJECT_ENDPOINT')
